# Dataset consolidado por paciente - Proyecto HTA (SIS)

Este cuaderno construye **un único `.csv` con una fila por paciente**, listo para la fase de *baseline*. Aplica los hallazgos de `01_eda.ipynb`.

```
|-------- 2022 y 2023 --------|---- 2024 ----|
 Variables (X): lo que ya pasó   Objetivo (y): ¿tuvo complicación?
```

| Decisión | Elección |
|---|---|
| Objetivo (y) | ACV/AIT, infarto, insuficiencia cardíaca, enfermedad renal o fallecimiento **nuevos en 2024** |
| Antecedentes (X) | Diabetes, dislipidemia, obesidad, fibrilación auricular y cardiopatía isquémica (2022-2023) |
| Columnas eliminadas | `FECATE_POST_FECFED` (no indica fallecimiento), `FECHA_CORTE`, `ULTIMO_MES_CONSUMOS` (constantes), `PROVINCIA`, `DISTRITO` (texto libre) |
| Duplicados | Se eliminan las filas idénticas de diagnósticos y procedimientos (y, por consistencia, de las otras tablas) |
| Ubicación | `es_lima_frecuente` (moda del departamento **de cada paciente**) y `altitud_msnm` (ubigeo más frecuente **de cada paciente**) |
| Variable nueva | `presion_pulso` = sistólica − diastólica |
| Escalado | **Se posterga.** Todas las variables quedan en sus unidades originales |
| Imputación | **No se imputa.** La presión arterial queda vacía (NaN) cuando nunca se midió, y `sin_medicion_pa` lo indica |

> **Nota sobre la tasa de la clase positiva.** El 15.52% de `01_eda.ipynb` contaba diabetes, dislipidemia y obesidad como complicaciones, y usaba como denominador a todos los pacientes con atenciones en 2024. Aquí esas condiciones son **antecedentes**, así que la tasa real del objetivo es bastante menor (≈5%). Se informa al final.

## Inicializar

In [1]:
import gc
import numpy as np
import pandas as pd
from pandas.api.types import union_categoricals

In [2]:
# --- Parámetros del trabajo ---
RUTA = ""            # carpeta de los CSV. "" = la misma carpeta del cuaderno.
FRAC = None          # fracción de pacientes (0.05 = 5 %). None = todos los pacientes
SEED = 42            # semilla 

FIN_OBS = 20231231   # Variables (X): atenciones hasta esta fecha
INI_RES = 20240101   # Objetivo (y): complicaciones desde esta fecha

CHUNK = 2_000_000    # filas por bloque al leer los CSV (para no agotar la memoria)
SALIDA = 'dataset_consolidado.csv' if FRAC is None else 'dataset_consolidado_muestra.csv'

## Lectura eficiente de los CSV
Las tablas completas suman ≈170 millones de filas. En corridas anteriores, `duplicated()` terminó con `MemoryError`. Por eso:
- se leen **solo las columnas necesarias** (`usecols`) y por bloques (`chunksize`);
- los textos repetidos (paciente, departamento, códigos) se guardan como `category`, que ocupa mucho menos memoria que el texto;
- cada bloque se puede **filtrar al leerlo**. Por ejemplo, solo se conservan las atenciones de los pacientes de la muestra.

In [3]:
def leer_csv(archivo, usecols, dtype, categorias=(), filtro=None):
    # Lee un CSV por bloques, aplica el filtro a cada bloque y convierte las columnas de texto a 'category'
    partes = []
    for bloque in pd.read_csv(RUTA + archivo, usecols=usecols, dtype=dtype, chunksize=CHUNK):
        if filtro is not None:
            bloque = filtro(bloque)
        for c in categorias:
            bloque[c] = bloque[c].astype('category')
        partes.append(bloque)
    # Unificar las categorías de todos los bloques para que pd.concat no las convierta en texto
    for c in categorias:
        todas = union_categoricals([p[c] for p in partes]).categories
        for p in partes:
            p[c] = p[c].cat.set_categories(todas)
    tabla = pd.concat(partes, ignore_index=True)
    del partes
    gc.collect()
    return tabla

## FUAS (una fila por atención)
Se descartan desde la lectura `FECATE_POST_FECFED`, `FECHA_CORTE`, `ULTIMO_MES_CONSUMOS`, `PROVINCIA`, `DISTRITO` y las columnas que no se usan. `UBIGEO` se lee como **texto** para no perder el cero inicial (`'021801'`, Áncash).

In [4]:
COLS_FUAS = ['CODIGO_ANONIMIZADO', 'SEXO', 'ID_REGISTRO_REL', 'FECHA_ATENCION', 'EDAD', 'UBIGEO',
             'DEPARTAMENTO', 'SERVICIO_PRESTACIONAL', 'DIAS_HOSP',
             'PRES_ART_SISTOLICA', 'PRES_ART_DIASTOLICA', 'DESTINO_ASEGURADO']
TIPOS_FUAS = {'CODIGO_ANONIMIZADO': str, 'SEXO': str, 'ID_REGISTRO_REL': 'int64', 'FECHA_ATENCION': 'int32',
              'EDAD': 'float32', 'UBIGEO': str, 'DEPARTAMENTO': str, 'SERVICIO_PRESTACIONAL': str,
              'DIAS_HOSP': 'float32', 'PRES_ART_SISTOLICA': 'float32', 'PRES_ART_DIASTOLICA': 'float32',
              'DESTINO_ASEGURADO': str}
CAT_FUAS = ['CODIGO_ANONIMIZADO', 'SEXO', 'UBIGEO', 'DEPARTAMENTO', 'SERVICIO_PRESTACIONAL', 'DESTINO_ASEGURADO']

if FRAC is not None:
    # Misma muestra que el cuaderno de Erich: pacientes en orden de aparición, sample(frac, random_state)
    unicos = []
    for bloque in pd.read_csv(RUTA + '1. HTA_FUAS.csv', usecols=['CODIGO_ANONIMIZADO'], chunksize=CHUNK):
        unicos.append(bloque['CODIGO_ANONIMIZADO'].drop_duplicates())
    pacientes_muestra = pd.concat(unicos).drop_duplicates().sample(frac=FRAC, random_state=SEED)
    pacientes_muestra = set(pacientes_muestra)
    del unicos
    filtro_fuas = lambda b: b[b['CODIGO_ANONIMIZADO'].isin(pacientes_muestra)]
else:
    filtro_fuas = None

fuas = leer_csv('1. HTA_FUAS.csv', COLS_FUAS, TIPOS_FUAS, CAT_FUAS, filtro_fuas)
fuas['UBIGEO'] = fuas['UBIGEO'].cat.rename_categories(lambda u: str(u).zfill(6))

# El hash del paciente (64 caracteres) se reemplaza por un número entero: ocupa mucho menos memoria.
# 'codigos_paciente' guarda la equivalencia para recuperar el hash al guardar el CSV.
codigos_paciente = fuas['CODIGO_ANONIMIZADO'].cat.categories
fuas['paciente'] = fuas['CODIGO_ANONIMIZADO'].cat.codes.astype('int32')
fuas = fuas.drop(columns='CODIGO_ANONIMIZADO')

print("Pacientes:", fuas['paciente'].nunique(), "| Atenciones:", len(fuas))
print(f"Memoria FUAS: {fuas.memory_usage(deep=True).sum() / 1e9:.2f} GB")

Pacientes: 1026473 | Atenciones: 24286707
Memoria FUAS: 0.92 GB


#### Corregir valores imposibles en FUAS
Es limpieza **fila por fila** con rangos fisiológicos fijos: no aprende nada de los datos, por lo que se puede hacer antes de separar train y test. Los valores imposibles pasan a NaN; **no se imputan**.

In [5]:
# Edad: entre 14 y 110 años
edad_mala = (fuas['EDAD'] < 14) | (fuas['EDAD'] > 110)
print("Edades imposibles:", edad_mala.sum())
fuas.loc[edad_mala, 'EDAD'] = np.nan

# Presión arterial: sistólica 60-260, diastólica 30-150 y sistólica > diastólica
pas = fuas['PRES_ART_SISTOLICA']
pad = fuas['PRES_ART_DIASTOLICA']
pa_valida = pas.between(60, 260) & pad.between(30, 150) & (pas > pad)
print("Presiones imposibles:", (pas.notna() & ~pa_valida).sum())
fuas.loc[~pa_valida, ['PRES_ART_SISTOLICA', 'PRES_ART_DIASTOLICA']] = np.nan

# Días de hospitalización: vacío = 0 (no hubo hospitalización), máximo 365
fuas['DIAS_HOSP'] = fuas['DIAS_HOSP'].fillna(0).clip(upper=365)

Edades imposibles: 397


Presiones imposibles: 2343


#### Columnas de apoyo en FUAS
- `pp` (presión de pulso de la atención) = sistólica − diastólica. Queda vacía si no se midió la presión.
- Ya **no** se crea `es_lima` por atención: la ubicación se resume más adelante con la moda de cada paciente.

In [6]:
pas = fuas['PRES_ART_SISTOLICA']
pad = fuas['PRES_ART_DIASTOLICA']

fuas['masculino']     = (fuas['SEXO'] == 'MASCULINO').astype('int8')
fuas['hospitalizado'] = (fuas['DIAS_HOSP'] > 0).astype('int8')
fuas['emergencia']    = fuas['SERVICIO_PRESTACIONAL'].str.upper().str.contains('EMERGENCIA').astype('int8')
fuas['fallecido']     = fuas['DESTINO_ASEGURADO'].str.upper().str.contains('FALLEC').astype('int8')
fuas['pa_alta']       = ((pas >= 140) | (pad >= 90)).astype('float32').where(pas.notna())   # NaN si no hubo medición
fuas['pp']            = pas - pad                                                           # NaN si no hubo medición

fuas = fuas.sort_values('FECHA_ATENCION', kind='stable')
print("Atenciones con fallecimiento registrado:", fuas['fallecido'].sum())

Atenciones con fallecimiento registrado: 31269


## Diagnósticos
Se eliminan las filas duplicadas exactas (mismo código y tipo en la misma atención) y se conservan solo los diagnósticos `DEFINITIVO` y `REPETIDO`.

In [7]:
ids_fuas = set(fuas['ID_REGISTRO_REL']) if FRAC is not None else None
filtro_ids = (lambda b: b[b['ID_REGISTRO_REL'].isin(ids_fuas)]) if FRAC is not None else None

diagnosticos = leer_csv('2. HTA_Diagnosticos.csv',
                        ['FECHA_CORTE', 'ULTIMO_MES_CONSUMOS', 'ID_REGISTRO_REL', 'CODDIA', 'TIPO_DIAGNOSTICO'],
                        {'FECHA_CORTE': 'int32', 'ULTIMO_MES_CONSUMOS': 'int32', 'ID_REGISTRO_REL': 'int64',
                         'CODDIA': str, 'TIPO_DIAGNOSTICO': str},
                        ['CODDIA', 'TIPO_DIAGNOSTICO'], filtro_ids)

antes = len(diagnosticos)
diagnosticos = diagnosticos.drop_duplicates()     # duplicados exactos (todas las columnas del archivo)
print(f"Diagnósticos duplicados eliminados: {antes - len(diagnosticos):,}")

# Limpiar espacios y quedarse con definitivos o repetidos
tipo = diagnosticos['TIPO_DIAGNOSTICO'].str.strip()
diagnosticos = diagnosticos.loc[tipo.isin(['DEFINITIVO', 'REPETIDO']).values, ['ID_REGISTRO_REL', 'CODDIA']]
cods_limpios = diagnosticos['CODDIA'].cat.categories.str.strip().str.upper()
if cods_limpios.is_unique:      # limpiar las categorías (rápido) si no se generan códigos repetidos
    diagnosticos['CODDIA'] = diagnosticos['CODDIA'].cat.rename_categories(cods_limpios)
else:
    diagnosticos['CODDIA'] = diagnosticos['CODDIA'].astype(str).str.strip().str.upper().astype('category')

# Agregar el paciente y la fecha a cada diagnóstico
dx = diagnosticos.merge(fuas[['ID_REGISTRO_REL', 'paciente', 'FECHA_ATENCION']], on='ID_REGISTRO_REL')
del diagnosticos, tipo
gc.collect()

grupos = {
    'hta':           ('I10', 'I11', 'I12', 'I13', 'I14', 'I15'),
    'acv':           ('I60', 'I61', 'I62', 'I63', 'I64', 'G45'),
    'iam':           ('I21', 'I22', 'I23'),
    'ic':            ('I50',),
    'erc':           ('N17', 'N18', 'N19'),
    'diabetes':      ('E10', 'E11', 'E12', 'E13', 'E14'),
    'dislipidemia':  ('E78',),
    'obesidad':      ('E66',),
    'fib_auricular': ('I48',),
    'cardiopatia':   ('I20', 'I24', 'I25'),
}
for nombre, codigos in grupos.items():
    dx[nombre] = dx['CODDIA'].str.startswith(codigos).astype(bool)

eventos = ['acv', 'iam', 'ic', 'erc']      # complicaciones que se quieren predecir (+ fallecimiento)
dx[['CODDIA'] + list(grupos)].head()

Diagnósticos duplicados eliminados: 4,489


,CODDIA,hta,acv,iam,ic,erc,diabetes,dislipidemia,obesidad,fib_auricular,cardiopatia
0,E119,False,False,False,False,False,True,False,False,False,False
1,N159,False,False,False,False,False,False,False,False,False,False
2,I10X,True,False,False,False,False,False,False,False,False,False
3,M198,False,False,False,False,False,False,False,False,False,False
4,K743,False,False,False,False,False,False,False,False,False,False


## Separar pasado (2022-2023) y futuro (2024)

In [8]:
fuas_obs = fuas[fuas['FECHA_ATENCION'] <= FIN_OBS]     # pasado -> variables
fuas_res = fuas[fuas['FECHA_ATENCION'] >= INI_RES]     # futuro -> objetivo

dx_obs = dx[dx['FECHA_ATENCION'] <= FIN_OBS]
dx_res = dx[dx['FECHA_ATENCION'] >= INI_RES]

print("Atenciones pasado:", len(fuas_obs), "| futuro:", len(fuas_res))

Atenciones pasado: 15017813 | futuro: 9268894


## Cohorte
`flujo` cuenta cuántos pacientes quedan en cada paso de selección. Se guarda como `flujo_cohorte.csv` para el informe; no es una variable del modelo.

In [9]:
n_total = fuas['paciente'].nunique()

# Pacientes con hipertensión en el pasado
con_hta = dx_obs.loc[dx_obs['hta'], 'paciente'].unique()

# Quitar los que ya tenían una complicación en el pasado
con_evento_previo = dx_obs.loc[dx_obs[eventos].any(axis=1), 'paciente'].unique()
cohorte = pd.Series(con_hta)
n_hta = len(cohorte)
cohorte = cohorte[~cohorte.isin(con_evento_previo)]
n_sin_previo = len(cohorte)

# Quedarse con los que tuvieron atenciones en 2024
cohorte = cohorte[cohorte.isin(fuas_res['paciente'].unique())]
n_final = len(cohorte)

flujo = pd.DataFrame({
    'Paso': ['Pacientes en los datos', 'Con hipertensión (2022-2023)',
             'Sin complicación previa', 'Con atenciones en 2024 (cohorte final)'],
    'Pacientes': [n_total, n_hta, n_sin_previo, n_final]
})
flujo

,Paso,Pacientes
0,Pacientes en los datos,1026473
1,Con hipertensión (2022-2023),632115
2,Sin complicación previa,552723
3,Con atenciones en 2024 (cohorte final),423314


## Variable objetivo (y)

In [10]:
# Complicaciones en 2024, por paciente (1 = tuvo, 0 = no)
tipos = dx_res.groupby('paciente', observed=True)[eventos].max().astype(int)
tipos['fallecido'] = fuas_res.groupby('paciente', observed=True)['fallecido'].max()

# Solo los pacientes de la cohorte; quienes no aparecen no tuvieron complicaciones
tipos = tipos.reindex(cohorte).fillna(0).astype(int)
tipos['y'] = tipos.max(axis=1)
tipos.sum()

acv           5617
iam           1031
ic            5509
erc           9713
fallecido     4309
y            22389
dtype: int64

## Variables desde FUAS
Además de las variables del cuaderno de Erich:
- **`es_lima_frecuente`**: en lugar del último departamento (`.last()`), se toma la **moda de `DEPARTAMENTO` en las atenciones 2022-2023 de cada paciente**. Si es `'LIMA'` vale 1; en otro caso, 0. Si hay empate, gana el departamento de la atención más reciente.
- **`altitud_msnm`**: se toma el **ubigeo más frecuente de cada paciente** (mismo criterio de empate) y se cruza con su altitud (ver la sección siguiente).
- **`presion_pulso`** (media) y **`presion_pulso_max`**: diferencia entre la sistólica y la diastólica. En hipertensos mayores de 50 años, una presión de pulso elevada indica rigidez arterial y predice infarto y ACV.

> Como sistólica y diastólica faltan siempre juntas, `presion_pulso` (media) es **exactamente** `pas_media − pad_media`. Para la regresión logística eso es colinealidad perfecta: al modelar, conviene quitar una de las tres (por ejemplo `pad_media`). `presion_pulso_max` sí aporta información nueva.
>
> `UBIGEO` y `DEPARTAMENTO` son la ubicación del **establecimiento** donde se atendió el paciente, no de su domicilio.

In [11]:
def moda_por_paciente(tabla, columna):
    # Valor más frecuente de 'columna' para CADA paciente; si hay empate, gana el de la atención más reciente
    conteo = (tabla.groupby(['paciente', columna], observed=True)
                   .agg(n=('FECHA_ATENCION', 'size'), ultima=('FECHA_ATENCION', 'max'))
                   .reset_index()
                   .sort_values(['paciente', 'n', 'ultima'], ascending=[True, False, False]))
    return conteo.drop_duplicates('paciente').set_index('paciente')[columna]

f = fuas_obs[fuas_obs['paciente'].isin(cohorte)]

X1 = f.groupby('paciente', observed=True).agg(
    edad=('EDAD', 'last'),                              # edad en la última atención
    masculino=('masculino', 'first'),
    n_atenciones=('ID_REGISTRO_REL', 'count'),
    n_emergencias=('emergencia', 'sum'),
    n_hospitalizaciones=('hospitalizado', 'sum'),
    dias_hosp=('DIAS_HOSP', 'sum'),
    n_mediciones_pa=('PRES_ART_SISTOLICA', 'count'),     # cuántas veces se midió la presión
    pas_media=('PRES_ART_SISTOLICA', 'mean'),
    pas_max=('PRES_ART_SISTOLICA', 'max'),
    pad_media=('PRES_ART_DIASTOLICA', 'mean'),
    pad_max=('PRES_ART_DIASTOLICA', 'max'),
    presion_pulso=('pp', 'mean'),
    presion_pulso_max=('pp', 'max'),
    prop_pa_alta=('pa_alta', 'mean'),                    # proporción de mediciones elevadas
)

dep_moda    = moda_por_paciente(f, 'DEPARTAMENTO')
ubigeo_moda = moda_por_paciente(f, 'UBIGEO')

X1.insert(2, 'es_lima_frecuente', (dep_moda.reindex(X1.index).astype(str) == 'LIMA').astype(int))

# Comparación con el criterio anterior (último departamento registrado)
dep_ultimo = f.groupby('paciente', observed=True)['DEPARTAMENTO'].last()
es_lima_ultimo = (dep_ultimo.reindex(X1.index).astype(str) == 'LIMA').astype(int)
print(f"Pacientes en que la moda y .last() difieren para Lima: {(es_lima_ultimo != X1['es_lima_frecuente']).sum():,} "
      f"de {len(X1):,}")
X1.head()

Pacientes en que la moda y .last() difieren para Lima: 6,572 de 423,314


,edad,masculino,es_lima_frecuente,n_atenciones,n_emergencias,n_hospitalizaciones,dias_hosp,n_mediciones_pa,pas_media,pas_max,pad_media,pad_max,presion_pulso,presion_pulso_max,prop_pa_alta
paciente,,,,,,,,,,,,,,,
0,75.0,1,0,19,0,0,0.0,8,108.750000,140.0,68.750000,90.0,40.000000,50.0,0.125000
1,55.0,0,0,37,4,0,0.0,28,135.892853,191.0,78.821426,99.0,57.071430,92.0,0.571429
3,63.0,0,1,51,1,0,0.0,12,123.083336,150.0,63.750000,75.0,59.333332,80.0,0.416667
9,63.0,0,0,40,1,0,0.0,23,128.608688,160.0,59.956520,76.0,68.652176,102.0,0.260870
17,63.0,0,0,5,2,1,6.0,3,126.666664,140.0,80.000000,100.0,46.666668,50.0,0.333333


#### Altitud del distrito (`altitud_msnm`)
Se eligió la librería **ubigeos-peru 0.2.3** (`pip install ubigeos-peru==0.2.3 --ignore-requires-python`, porque declara Python < 3.13). Sin embargo, su función `get_metadato(..., level='distritos', key='altitud')` busca la altitud **por nombre del distrito**, no por código. Por eso los distritos con el mismo nombre reciben la misma altitud: por ejemplo, los 10 distritos *Santa Rosa* reciben 288 m, incluido Santa Rosa de Puno, que está a ≈3,980 m.

Para evitarlo se usa la base de datos de la misma librería, `databases/ubigeo_inei_2025.csv` de su repositorio ([github.com/MichaelSuarez0/ubigeos_peru](https://github.com/MichaelSuarez0/ubigeos_peru)). Esa base tiene la altitud **por código ubigeo INEI**. Los ubigeos del SIS son INEI (Lima = `15xxxx`).

In [12]:
geo = pd.read_csv(RUTA + 'ubigeo_inei_2025.csv', sep=';', dtype={'ubigeo': str})
altitud = geo.set_index('ubigeo')['altitud']
print("Control:", altitud.loc[['150101', '080101', '210101', '210504']].to_dict(),
      "(Lima, Cusco, Puno, Santa Rosa-Puno)")

X1.insert(3, 'altitud_msnm', ubigeo_moda.reindex(X1.index).astype(str).map(altitud))

sin_altitud = ubigeo_moda.astype(str)[~ubigeo_moda.astype(str).isin(altitud.index)]
print(f"Pacientes cuyo ubigeo no está en la tabla de altitudes: {len(sin_altitud):,}",
      f"| ubigeos: {sorted(sin_altitud.unique())[:20]}")
X1['altitud_msnm'].describe().round(1)

Control: {'150101': 162.42, '080101': 3439.0, '210101': 3847.6, '210504': 3980.6} (Lima, Cusco, Puno, Santa Rosa-Puno)
Pacientes cuyo ubigeo no está en la tabla de altitudes: 0 | ubigeos: []


count    423314.0
mean        914.9
std        1227.6
min           5.2
25%          98.4
50%         196.8
75%        1640.3
max        4705.4
Name: altitud_msnm, dtype: float64

## Variables desde Diagnósticos (antecedentes)

In [13]:
d = dx_obs[dx_obs['paciente'].isin(cohorte)]

comorbilidades = ['diabetes', 'dislipidemia', 'obesidad', 'fib_auricular', 'cardiopatia']
X2 = d.groupby('paciente', observed=True)[comorbilidades].max().astype(int)

# Cantidad de diagnósticos distintos
X2['n_diagnosticos'] = d.drop_duplicates(['paciente', 'CODDIA']).groupby('paciente', observed=True).size()

del dx, dx_obs, dx_res, d
gc.collect()
X2.head()

,diabetes,dislipidemia,obesidad,fib_auricular,cardiopatia,n_diagnosticos
paciente,,,,,,
0,1,1,0,0,0,12
1,1,0,0,0,1,25
3,1,1,0,0,0,26
9,1,1,0,0,0,28
17,1,0,0,0,0,11


## Variables desde Medicamentos, Procedimientos e Insumos
Solo se leen las filas de las **atenciones 2022-2023 de la cohorte**. Se eliminan los duplicados exactos y se recortan los montos absurdos (errores de digitación) al percentil 99.9, como en el cuaderno de Erich. Se mantienen los agregados `n_*` y `gasto_*`: no se pivotan códigos.

In [14]:
atenciones_cohorte = f[['ID_REGISTRO_REL', 'paciente']].copy()
ids_cohorte = set(atenciones_cohorte['ID_REGISTRO_REL'])

def resumir(archivo, columna_codigo, nombre, contar_distintos):
    # Cuenta cuántos items recibió cada paciente (en el pasado) y suma el gasto
    t = leer_csv(archivo,
                 ['FECHA_CORTE', 'ULTIMO_MES_CONSUMOS', 'ID_REGISTRO_REL', 'CODDIA', columna_codigo,
                  'CANTIDAD_ENTREGADA', 'VALOR_NETO'],
                 {'FECHA_CORTE': 'int32', 'ULTIMO_MES_CONSUMOS': 'int32', 'ID_REGISTRO_REL': 'int64',
                  'CODDIA': str, columna_codigo: str, 'CANTIDAD_ENTREGADA': 'int64', 'VALOR_NETO': 'float64'},
                 ['CODDIA', columna_codigo],
                 lambda b: b[b['ID_REGISTRO_REL'].isin(ids_cohorte)])
    antes = len(t)
    t = t.drop_duplicates()
    print(f"{nombre}: {antes - len(t):,} duplicados eliminados")

    t = t.merge(atenciones_cohorte, on='ID_REGISTRO_REL')
    t['VALOR_NETO'] = t['VALOR_NETO'].clip(upper=t['VALOR_NETO'].quantile(0.999))

    if contar_distintos:     # items distintos (ej. medicamentos diferentes)
        n = t.drop_duplicates(['paciente', columna_codigo]).groupby('paciente').size()
    else:                    # total de items
        n = t.groupby('paciente').size()

    resumen = pd.DataFrame({f'n_{nombre}': n, f'gasto_{nombre}': t.groupby('paciente')['VALOR_NETO'].sum()})
    del t
    gc.collect()
    return resumen

X3 = resumir('3. HTA_Medicamentos.csv',   'COD_MEDICAMENTO',    'medicamentos',   contar_distintos=True)
X4 = resumir('4. HTA_Procedimientos.csv', 'COD_PROCEDIMIIENTO', 'procedimientos', contar_distintos=False)
X5 = resumir('5. HTA_Insumos.csv',        'COD_INSUMO',         'insumos',        contar_distintos=False)
X3.head()

medicamentos: 0 duplicados eliminados


procedimientos: 3,818 duplicados eliminados


insumos: 0 duplicados eliminados


,n_medicamentos,gasto_medicamentos
paciente,,
0,8,12.000
1,24,55.666
3,27,419.337
9,11,95.690
17,15,64.616


## Unir todo en una sola tabla
- Si el paciente no aparece en una tabla, significa que no recibió nada: se rellena con **0**.
- La presión arterial y la presión de pulso **no se imputan**. Quedan en NaN para los pacientes a los que nunca se les midió, y `sin_medicion_pa` = 1 lo indica. La imputación se hará en la fase de modelado, ajustada **solo con train**.
- No se aplica clipping IQR, `log1p`, escalado, separación train/test ni SMOTE.

In [15]:
df = X1.join(X2).join(X3).join(X4).join(X5)

cols_cero = list(X2.columns) + list(X3.columns) + list(X4.columns) + list(X5.columns)
df[cols_cero] = df[cols_cero].fillna(0)
cols_enteras = [c for c in cols_cero if not c.startswith('gasto_')]
df[cols_enteras] = df[cols_enteras].astype(int)

df['polifarmacia']    = (df['n_medicamentos'] >= 5).astype(int)    # 5 o más medicamentos distintos
df['sin_medicion_pa'] = (df['n_mediciones_pa'] == 0).astype(int)   # nunca se le midió la presión

df['y'] = tipos['y'].reindex(df.index).astype(int)

print("Pacientes:", df.shape[0], "| Columnas (incluida y):", df.shape[1])
df.head()

Pacientes: 423314 | Columnas (incluida y): 31


,edad,masculino,es_lima_frecuente,altitud_msnm,n_atenciones,n_emergencias,n_hospitalizaciones,dias_hosp,n_mediciones_pa,pas_media,...,n_diagnosticos,n_medicamentos,gasto_medicamentos,n_procedimientos,gasto_procedimientos,n_insumos,gasto_insumos,polifarmacia,sin_medicion_pa,y
paciente,,,,,,,,,,,,,,,,,,,,,
0,75.0,1,0,26.578,19,0,0,0.0,8,108.750000,...,12,8,12.000,36,0.00,1,0.000,1,0,0
1,55.0,0,0,2452.990,37,4,0,0.0,28,135.892853,...,25,24,55.666,46,157.17,13,0.890,1,0,1
3,63.0,0,1,137.800,51,1,0,0.0,12,123.083336,...,26,27,419.337,88,865.71,55,149.788,1,0,0
9,63.0,0,0,136.760,40,1,0,0.0,23,128.608688,...,28,11,95.690,91,292.78,14,0.556,1,0,0
17,63.0,0,0,43.250,5,2,1,6.0,3,126.666664,...,11,15,64.616,33,517.43,12,23.028,1,0,0


## Verificaciones

In [16]:
print(f"Tasa de la clase positiva (y=1): {df['y'].mean():.2%} "
      f"({df['y'].sum():,} de {len(df):,}) | desbalance {(1 - df['y'].mean()) / df['y'].mean():.1f}:1")

# Presión de pulso
pp = df['presion_pulso'].dropna()
print(f"Presión de pulso: min {pp.min():.1f}, max {df['presion_pulso_max'].max():.1f} | "
      f"todas > 0: {(pp > 0).all()} | "
      f"igual a pas_media - pad_media: {np.allclose(pp, (df['pas_media'] - df['pad_media']).dropna(), atol=1e-3)}")

# Faltantes: solo deberían faltar las variables de presión (pacientes sin medición) y alguna altitud
faltantes = df.isna().sum()
print("\nFaltantes por columna:")
print(faltantes[faltantes > 0].to_string())
print("Pacientes sin medición de PA:", df['sin_medicion_pa'].sum())

Tasa de la clase positiva (y=1): 5.29% (22,389 de 423,314) | desbalance 17.9:1
Presión de pulso: min 2.0, max 200.0 | todas > 0: True | igual a pas_media - pad_media: True

Faltantes por columna:
edad                     3
pas_media            16451
pas_max              16451
pad_media            16451
pad_max              16451
presion_pulso        16451
presion_pulso_max    16451
prop_pa_alta         16451
Pacientes sin medición de PA: 16451


In [17]:
# Unidades originales (sin escalar): edad ≈ 65, PAS ≈ 120, altitud en metros, gasto en soles
df.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
edad,423311.0,65.32,14.26,16.00,56.00,67.00,76.00,110.00
masculino,423314.0,0.33,0.47,0.00,0.00,0.00,1.00,1.00
es_lima_frecuente,423314.0,0.28,0.45,0.00,0.00,0.00,1.00,1.00
altitud_msnm,423314.0,914.92,1227.55,5.18,98.40,196.80,1640.30,4705.40
n_atenciones,423314.0,20.45,18.28,1.00,8.00,16.00,28.00,902.00
n_emergencias,423314.0,1.05,2.53,0.00,0.00,0.00,1.00,343.00
n_hospitalizaciones,423314.0,0.17,0.52,0.00,0.00,0.00,0.00,31.00
dias_hosp,423314.0,1.38,10.71,0.00,0.00,0.00,0.00,991.00
n_mediciones_pa,423314.0,10.31,9.38,0.00,4.00,8.00,15.00,276.00
pas_media,406863.0,123.99,16.00,60.00,113.08,122.00,132.64,260.00


## Guardar

In [18]:
df.index = codigos_paciente[df.index]   # recuperar el código anonimizado del paciente
df.index.name = 'paciente'
df.to_csv(SALIDA)
flujo.to_csv('flujo_cohorte.csv' if FRAC is None else 'flujo_cohorte_muestra.csv', index=False)
print("Guardado:", SALIDA, df.shape)

Guardado: dataset_consolidado.csv (423314, 31)


## Resumen
1. **Una fila por paciente** de la cohorte (hipertensos en 2022-2023, sin ACV, IAM, IC ni ERC previos y con atenciones en 2024).
2. **Objetivo `y`**: ACV/AIT, IAM, IC, ERC o fallecimiento en 2024. La clase positiva es minoritaria (ver verificación), así que habrá que usar `class_weight`, SMOTE solo en train y métricas como Recall, F1 y AUC.
3. **Antecedentes**: diabetes, dislipidemia, obesidad, fibrilación auricular y cardiopatía isquémica.
4. **Ubicación**: `es_lima_frecuente` (moda del departamento de cada paciente) y `altitud_msnm` (ubigeo más frecuente de cada paciente). Ambas corresponden a la ubicación del establecimiento de salud.
5. **Unidades originales**: sin escalado, sin `log1p` y sin imputación. Todo eso se hará en la fase de modelado, ajustado solo con train.